In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
! # Make sure to enable the ipycanvas/ other widgets
!jupyter labextension enable widgetsnbextension

In [ ]:
import torch
from popari.model import Popari
from popari import pl, tl
from popari.util import concatenate
import scanpy as sc
from matplotlib import pyplot as plt
from pathlib import Path

from popari.train import Trainer, BatchBlendTrainer, TrainParameters
import numpy as np
import scanpy as sc
from scib_metrics.benchmark import Benchmarker, BioConservation, BatchCorrection
import anndata as ad
import os

In [ ]:
directory = "simulated_batch_data/input/"

all_files = []
for root, dirs, files in os.walk(directory):
    for file in files:
        file_path = os.path.join(root, file)
        all_files.append(file_path)

print(all_files)

In [ ]:
lambda_Sigma_x_inv=1e-4
lambda_Sigma_bar=1e-4
torch_context={
    "dtype": torch.float64,
    "device": "cuda:0"
}
K = 11
seed = 42


for file in all_files:
    print(file)

    original_adata = ad.read_h5ad(file)
    
    nmf_preiterations = 10
    num_iterations = 50
    model = Popari(                                                              
        K=K,                                                                        
        dataset_path=file,                                                                        
        lambda_Sigma_x_inv=lambda_Sigma_x_inv,
        spatial_affinity_mode="differential lookup",
        spatial_affinity_groups={
            "progenitor": ["progenitor"],
            "layer": ["layer"],
        },
        prior_x_modes = ["exponential shared fixed"]*2, 
        initial_context=torch_context,                                              
        torch_context=torch_context,
        initialization_method="leiden",
        hierarchical_levels=1,
        verbose=1,                                                                  
        random_state=seed,
    )                  
    
    train_parameters = TrainParameters(
        nmf_iterations=nmf_preiterations,
        iterations=num_iterations,
        savepath=(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/trained_{num_iterations}_iterations.h5ad"),
    )
    
    trainer = Trainer(
        parameters=train_parameters,
        model=model,
        verbose=True,
    )
    
    trainer.train()

    merged_dataset = concatenate(model.datasets)
    original_adata.obsm["X_popari"] = merged_dataset.obsm["X"]
    



    
    
    nmf_preiterations = 50
    num_iterations = 0
    model_2 = Popari(                                                              
        K=K,                                                                        
        dataset_path=file,                                                                        
        lambda_Sigma_x_inv=lambda_Sigma_x_inv,
        spatial_affinity_mode="differential lookup",
        spatial_affinity_groups={
            "progenitor": ["progenitor"],
            "layer": ["layer"],
        },
        initial_context=torch_context,                                              
        torch_context=torch_context,
        embedding_acceleration_trick=False,
        initialization_method="leiden",
        hierarchical_levels=1,
        prior_x_modes = ["cross_dataset_average", "cross_dataset_average"], 
        verbose=1,                                                                  
        random_state=seed,
        batch_effect_correction="joint_metagenes",
    )      
    
    
    train_parameters_2 = TrainParameters(
        nmf_iterations=nmf_preiterations,
        iterations=num_iterations,
        savepath=(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/trained_{num_iterations}_iterations_batch.h5ad"),
    )
    
    trainer_2 = BatchBlendTrainer(
        parameters=train_parameters_2,
        model=model_2,
        verbose=1,
    )
    
    trainer_2.train()  

    for dataset in model_2.datasets:
        batch_effect_key = list(dataset.uns['batch_effect'].keys())[0]
        batch_effect = dataset.uns['batch_effect'][batch_effect_key]
        original_data = dataset.obsm['X']
        batch_corrected_data = original_data + batch_effect
        dataset.obsm['X_added_batch_blend'] = batch_corrected_data

    merged_dataset_2 = concatenate(model_2.datasets)
    original_adata.obsm["X_batchblend"] = merged_dataset_2.obsm["X"]
    original_adata.obsm["X_added_batch_blend"] = merged_dataset_2.obsm["X_added_batch_blend"]


    save_path = file.replace("input", "output")    
    original_adata.write(save_path)
    print(save_path)

In [ ]:
for file in all_files:
    print(f"results for {file}")
    adata, popari, batch_blend  = models[file]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(adata, use_rep="ground_truth_X")
    sc.tl.umap(adata)
    sc.pl.umap(adata, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (Ground Truth)")
    sc.pl.umap(adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Ground Truth)")
    plt.tight_layout()
    fig.show()

    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(adata, use_rep="X_popari")
    sc.tl.umap(adata)
    sc.pl.umap(adata, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (Popari)")
    sc.pl.umap(adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Popari)")
    plt.tight_layout()
    fig.show()

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(adata, use_rep="X_batchblend")
    sc.tl.umap(adata)
    sc.pl.umap(adata, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (BatchBlend)")
    sc.pl.umap(adata, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (BatchBlend)")
    plt.tight_layout()
    fig.show()

    np.savetxt(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/popari_" + str(file.split("0_")[-1:]) + ".csv", adata.obsm['X_popari'], delimiter=',', fmt='%d')
    np.savetxt(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/batchblend_" + str(file.split("0_")[-1:]) + ".csv", adata.obsm['X_batchblend'], delimiter=',', fmt='%d')

    for dataset in batch_blend.datasets:
        batch_effect_key = list(dataset.uns['batch_effect'].keys())[0]
        batch_effect = dataset.uns['batch_effect'][batch_effect_key]
        original_data = dataset.obsm['X']
        batch_corrected_data = original_data + batch_effect
        dataset.obsm['X_added_batch_blend'] = batch_corrected_data
    print(original_data, batch_effect, batch_corrected_data)
    
    merged_dataset = concatenate(batch_blend.datasets)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sc.pp.neighbors(merged_dataset, use_rep="X_added_batch_blend")
    sc.tl.umap(merged_dataset)
    sc.pl.umap(merged_dataset, color="cell_type", alpha=0.5, ax=ax1, show=False, title="Cell types (Batch Blend)")
    sc.pl.umap(merged_dataset, color="batch", alpha=0.5, ax=ax2, show=False, title="Batches (Batch Blend)")
    plt.tight_layout()
    fig.show()


    np.savetxt(f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/batch_plus_x_" + str(file.split("0_")[-1:]) + ".csv", merged_dataset.obsm['X_added_batch_blend'], delimiter=',', fmt='%d')
    
    bm = Benchmarker(
        adata,
        batch_key="batch",
        label_key="cell_type",
        bio_conservation_metrics=BioConservation(),
        batch_correction_metrics=BatchCorrection(),
        embedding_obsm_keys=["X_popari", "X_batchblend"],
        n_jobs=1,
    )
    bm.benchmark()
    bm.plot_results_table()
    bm.plot_results_table(min_max_scale=False)
    
    df = bm.get_results(min_max_scale=False)
    df.to_csv("/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/data_" + str(file.split("0_")[-1:]) + ".csv")

    path = f"/home/raehashs/batch_effect_correction/popari/simulated_batch_data/output/data_" + str(file.split("0_")[-1:]) + ".h5ad"
    adata.write(path)

In [ ]:
from popari._dataset_utils import _plot_all_embeddings

total_metagenes = 11
fov = 1
size= 8
for val in models:
    adata, popari, batch_blend = models[val]

    fig, axes = plt.subplots(3, total_metagenes, sharex=True, sharey=True, tight_layout=True, dpi=300, figsize=(total_metagenes, 3))
    _plot_all_embeddings.__wrapped__(popari.hierarchy[0].datasets[fov], size=size, embedding_key="ground_truth_X", colorbar=False, fig=fig, ax=axes[0, :].flat, edgecolors='none')
    _plot_all_embeddings.__wrapped__(popari.hierarchy[0].datasets[fov], size=size, colorbar=False, fig=fig, ax=axes[1, :K].flat, edgecolors='none')
    _plot_all_embeddings.__wrapped__(batch_blend.hierarchy[0].datasets[fov], size=size, colorbar=False, fig=fig, ax=axes[2, :K].flat, edgecolors='none')

    break

In [ ]:
from popari.util import unconcatenate
data = ad.read_h5ad("simulated_batch_data/output/batch_effect_3_with_50_percent/processed_dataset_13.h5ad")
data

In [ ]:
print(data.obsm['X_added_batch_blend'].shape,
data.obsm['X_batchblend'].shape,
data.obsm['X_popari'].shape, data.obsm['ground_truth_X'].shape)